# Hybrid Search + Reranking (Qdrant)
Testing hybrid search (Dense + BM25) and Cross-Encoder re-ranking on support docs.

### 1. Create support docs dataset

In [1]:
with open("support.txt", "w", encoding="utf-8") as file:
    file.write("""You can request a full refund within 14 days of purchase.
To get your money back, contact our billing team through the app.
Payments are processed securely using Stripe.
Error ERR-4012 means the password you entered is incorrect.
Error ERR-5003 appears when the server is under maintenance.
If you see ERR-7781, clear the app cache and restart CloudNote.
CloudNote Pro costs 5 dollars per month and includes unlimited storage.
The free plan of CloudNote gives you 2 GB of storage.
You can export all your notes as PDF or Markdown files.
To reset your password, click "Forgot password" on the login screen.
If you cannot log in, make sure your email address is verified.
CloudNote syncs your notes automatically across all your devices.
Sync may stop working if you are offline for a long time.
Two-factor authentication can be enabled from Settings > Security.
Our support team is available from 9 AM to 5 PM, Sunday to Thursday.
Deleted notes stay in the Trash folder for 30 days before permanent removal.
You can share a note with a teammate by sending them an invite link.
CloudNote works on Windows, macOS, Android, and iOS.""")

### 2. Connect to Qdrant & create hybrid collection

In [2]:
from qdrant_client import QdrantClient,models

client = QdrantClient(url="http://localhost:6333")

print("connection done")

connection done


/Users/saherelshewikh/projects/AI_Engineer/steps/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
collection_name = "hybrid_search_practice"

if not client.collection_exists(collection_name = collection_name):
    client.create_collection(
        collection_name= collection_name,
        vectors_config= {
            "dense":models.VectorParams(
             size=384,
            distance=models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            "sparse":models.SparseVectorParams(
                  modifier=models.Modifier.IDF ,
              
            )
        }
    )

### 3. Embed documents and upsert
- Dense: `multilingual-e5-small` (384d)
- Sparse: `Qdrant/bm25` (FastEmbed)

In [4]:
with open("support.txt", encoding="utf-8") as f:
    docs = [line.strip() for line in f if line.strip()]
# حساب متوسط عدد الكلمات في كل جملة
avg = sum(len(doc.split()) for doc in docs) / len(docs)
print(f"Total documents: {len(docs)}")
print(f"Average words per doc: {avg:.2f}")    

Total documents: 18
Average words per doc: 10.72


In [5]:
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer("intfloat/multilingual-e5-small")
dense_vectors = dense_model.encode([f"passage: {doc}" for doc in docs])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10267.76it/s]


In [6]:
client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=i,  
            payload={"text": doc},
            vector={
                "dense": dense_vectors[i].tolist(),
                "sparse": models.Document(
                    text=doc,
                    model="Qdrant/bm25",
                    options={"avg_len": avg}
                )
            }
        )
        for i, doc in enumerate(docs)
    ]
)

print(f"Successfully upserted {len(docs)} documents.")


Successfully upserted 18 documents.


### 4. Dense vs Sparse (BM25) comparison
Testing 3 query types:
- Semantic intent -> dense should win
- Exact error code -> sparse should win
- Mixed query -> hybrid candidate

In [7]:
questions = [
    "How can I get my money back?",      # معنوي، والـ dense المفروض يكسب
    "What does ERR-7781 mean?",          # كود، والـ sparse المفروض يكسب
    "Why is my sync not working?",       # مختلط
]

In [8]:
def dense_search(query) -> list[models.ScoredPoint]:
    query_vector = dense_model.encode(f"query: {query}").tolist()
    response = client.query_points(
        collection_name= collection_name,
        using = "dense",
        query = query_vector,
        limit = 3
    )
    return response.points

def sparse_search(query) -> list[models.ScoredPoint]:
 
    response = client.query_points(
        collection_name= collection_name,
        using = "sparse",
        query = models.Document(
            text=query,
            model="Qdrant/bm25"
        ),
        limit = 3
    )
    return response.points    

In [10]:
for q in questions:
    print(f"\n question: '{q}'")

    print("\n--- Dense Search ---")
    for r in dense_search(q):
        print(f"  ({r.score:.4f}) -> {r.payload['text']}")

    print("\n--- [Sparse (BM25) Search] ---")    
    for r in sparse_search(q):
        print(f"  ({r.score:.4f}) -> {r.payload['text']}")




 question: 'How can I get my money back?'

--- Dense Search ---
  (0.8694) -> To get your money back, contact our billing team through the app.
  (0.8186) -> You can export all your notes as PDF or Markdown files.
  (0.8179) -> You can request a full refund within 14 days of purchase.

--- [Sparse (BM25) Search] ---
  (8.8777) -> To get your money back, contact our billing team through the app.

 question: 'What does ERR-7781 mean?'

--- Dense Search ---
  (0.8644) -> If you see ERR-7781, clear the app cache and restart CloudNote.
  (0.8057) -> Error ERR-5003 appears when the server is under maintenance.
  (0.7915) -> Error ERR-4012 means the password you entered is incorrect.

--- [Sparse (BM25) Search] ---
  (4.9309) -> Error ERR-4012 means the password you entered is incorrect.
  (4.7210) -> If you see ERR-7781, clear the app cache and restart CloudNote.
  (2.0634) -> Error ERR-5003 appears when the server is under maintenance.

 question: 'Why is my sync not working?'

--- Dense S

### 5. Hybrid Search (RRF)
Merging dense and sparse results with Reciprocal Rank Fusion.

In [15]:
def hybrid_search(query, limit=10) -> list[models.ScoredPoint]:
    query_vector = dense_model.encode(f"query: {query}").tolist()
    response = client.query_points(
        collection_name = collection_name,
        prefetch=[
         models.Prefetch(
            query =  query_vector ,
            using="dense",
            limit = limit
         ),
         models.Prefetch(
            query = models.Document(
                text = query,
                model= "Qdrant/bm25"
            ),
            using= "sparse",
            limit = limit,
         ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit = limit 
    )
    return response.points

In [16]:
for q in questions:
    print(f"\n question: '{q}'")

    print("\n--- Hybrid Search ---")
    for r in hybrid_search(q):
        print(f"  ({r.score:.4f}) -> {r.payload['text']}")


 question: 'How can I get my money back?'

--- Hybrid Search ---
  (1.0000) -> To get your money back, contact our billing team through the app.
  (0.3333) -> You can export all your notes as PDF or Markdown files.
  (0.2500) -> You can request a full refund within 14 days of purchase.
  (0.2000) -> To reset your password, click "Forgot password" on the login screen.
  (0.1667) -> Our support team is available from 9 AM to 5 PM, Sunday to Thursday.
  (0.1429) -> If you see ERR-7781, clear the app cache and restart CloudNote.
  (0.1250) -> If you cannot log in, make sure your email address is verified.
  (0.1111) -> CloudNote Pro costs 5 dollars per month and includes unlimited storage.
  (0.1000) -> Error ERR-4012 means the password you entered is incorrect.
  (0.0909) -> Two-factor authentication can be enabled from Settings > Security.

 question: 'What does ERR-7781 mean?'

--- Hybrid Search ---
  (0.8333) -> If you see ERR-7781, clear the app cache and restart CloudNote.
  (0.7500

### 6. Re-ranking with Cross-Encoder
Re-ranking top hybrid candidates using `cross-encoder/ms-marco-MiniLM-L-6-v2`.

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6807.57it/s]


In [ ]:
def rerank_search(query: str, top_k: int = 3):
    points = hybrid_search(query)
    texts = [p.payload["text"] for p in points]
    ranked = reranker.rank(query, texts, top_k=top_k)  # return [ {"corpus_id": 0, "score": 8.45}, ]
    return [(r["score"], points[r["corpus_id"]]) for r in ranked]

In [22]:
for q in questions:
    print(f"\nQuestion: '{q}'")
    for score, point in rerank_search(q):
        print(f"  [{score:.4f}] -> {point.payload['text']}")


Question: 'How can I get my money back?'
  [7.2906] -> To get your money back, contact our billing team through the app.
  [-4.6228] -> You can request a full refund within 14 days of purchase.
  [-10.6715] -> You can export all your notes as PDF or Markdown files.

Question: 'What does ERR-7781 mean?'
  [3.8639] -> If you see ERR-7781, clear the app cache and restart CloudNote.
  [1.6114] -> Error ERR-4012 means the password you entered is incorrect.
  [-1.0543] -> Error ERR-5003 appears when the server is under maintenance.

Question: 'Why is my sync not working?'
  [6.4462] -> Sync may stop working if you are offline for a long time.
  [-6.9762] -> CloudNote syncs your notes automatically across all your devices.
  [-10.9408] -> Error ERR-5003 appears when the server is under maintenance.
